# 05 - FISPACT activation studies

Solutions of the exercises. The setup cell below rebuilds the few objects they need, so this
notebook runs on its own.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from f4enix.core.constants import TIME_UNITS
from f4enix.core.egroups import GROUP_STRUCTURES
from f4enix.core.irradiation import IrradiationScenario, Pulse
from f4enix.core.spectra import Spectra
from f4enix.input.fispact_inp import FispactInp
from f4enix.input.MCNPinput import Input
from f4enix.output.fispact_legacy_out import FispactOutput
from f4enix.output.mctal import Mctal

DATA = Path("..") / "data"
FISP = DATA / "fispact"
OUT_FISP = Path("..") / "outputs" / "fispact"
OUT_FISP.mkdir(parents=True, exist_ok=True)

mctal = Mctal(DATA / "iter_1d" / "mctal")
mctal.remove_totals(4)
flux = mctal.tallydata[4]
ebins = GROUP_STRUCTURES["VITAMIN-J-175"]
inp = Input.from_input(DATA / "iter_1d" / "ITER_1D.i")

pulsed = IrradiationScenario(
    [Pulse(2, 0.05, TIME_UNITS.YEAR)] + [Pulse(3290, 0), Pulse(400, 1.0)] * 17,
    name="two_years_then_pulses",
)
pulsed.set_cooling_times([(1, TIME_UNITS.HOUR), (1, TIME_UNITS.DAY), (1, TIME_UNITS.YEAR)])

scenario = IrradiationScenario.from_fispact(FISP / "inp_fispact.i")
scenario.set_cooling_times([
    (1, TIME_UNITS.DAY), (11.6, TIME_UNITS.DAY), (30, TIME_UNITS.DAY),
    (180, TIME_UNITS.DAY), (1, TIME_UNITS.YEAR), (10, TIME_UNITS.YEAR),
])
fispact_out = FispactOutput(FISP / "testSS.out", scenario.cooling_labels)


def add_dose_and_uncert(path: Path) -> str:
    """Add DOSE 1 and UNCERT 2 to an input written by FispactInp (only once)."""
    text = path.read_text()
    if "\nDOSE " not in text:
        text = text.replace(
            "<< INITIALIZATION PHASE >>",
            "<< INITIALIZATION PHASE >>\n"
            "<< contact dose rate, uncertainties and pathways >>\n"
            "DOSE 1\n"
            "UNCERT 2",
        )
        path.write_text(text)
    return text

c:\F4EMiniforge3\envs\f4enix_workshop\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---

# Exercises

## Exercise 1 - an inventory input for the steel behind the armour

Cell 55 sits just behind the tungsten armour, and it is made of `M11`, the 316L(N)-IG
stainless steel. Prepare its activation run with the `pulsed` scenario of section 5.

1. extract the spectrum of cell 55 from `flux`, keep its total flux in `steel_flux` and build
   a normalised `Spectra` called `steel_spectrum`, named `"ITER1D_C55"`;
2. write its `fluxes` file to `OUT_FISP`;
3. build a `FispactInp` called `steel_inp` with 1 kg of `M11` in `"mass"` style and the
   `pulsed` scenario at the flux of cell 55;
4. make it read the collapsed cross sections and the condensed decay data (`GETXS 0`,
   `GETDECAY 0`), and save it to `steel_file = OUT_FISP / "steel_c55.i"`;
5. add `DOSE 1` and `UNCERT 2` to the saved file.

**Hints**
* section 2 does steps 1 and 2 for cell 53: `flux[flux["Cells"] == 55]`, then
  `Spectra(ebins, values / values.sum(), name=...)` and `to_fispact_fluxes(OUT_FISP)`;
* the material is `inp.mat_section["M11"]`;
* `add_irradiation_scenario(pulsed, norm=steel_flux)` - the pulses are fractions of full power;
* `readXSData(0)` and `readDecayData(condense=False)` on `steel_inp.inp`;
* `add_dose_and_uncert` of section 6 works on any file saved by `FispactInp`.

In [2]:
# 1: same recipe as cell 53, the total is the flux at full power
values = flux[flux["Cells"] == 55]["Value"].to_numpy()
steel_flux = values.sum()
steel_spectrum = Spectra(ebins, values / steel_flux, name="ITER1D_C55")

# 2: the file is named after the spectrum
steel_spectrum.to_fispact_fluxes(OUT_FISP)

# 3: the scenario is in fractions of full power, norm makes it a flux
steel_inp = FispactInp()
steel_inp.add_material(inp.mat_section["M11"], style="mass", mass=1.0)
steel_inp.add_irradiation_scenario(pulsed, norm=steel_flux)

# 4: collapse and condense run separately, the inventory only reads them
steel_inp.inp.readXSData(0)
steel_inp.inp.readDecayData(condense=False)
steel_file = OUT_FISP / "steel_c55.i"
steel_inp.save(steel_file)

# 5: the two keywords pypact cannot write
add_dose_and_uncert(steel_file)

print(f"flux in cell 55 : {steel_flux:.3e} n/cm2/s")
print(f"flux in cell 53 : {flux[flux['Cells'] == 53]['Value'].sum():.3e} n/cm2/s")

flux in cell 55 : 3.357e+14 n/cm2/s
flux in cell 53 : 3.683e+14 n/cm2/s


In [3]:
# --- self-check, do not modify ---
expected_flux = flux[flux["Cells"] == 55]["Value"].sum()
assert abs(steel_flux / expected_flux - 1) < 1e-9, (
    f"steel_flux is {steel_flux:.4e}, the total flux of cell 55 is {expected_flux:.4e}"
)
assert steel_spectrum.name == "ITER1D_C55", f"the spectrum is named {steel_spectrum.name!r}"
assert abs(steel_spectrum.spectra_values.sum() - 1) < 1e-6, "the spectrum is not normalised"
reread = Spectra.from_fispact(ebins, OUT_FISP / "ITER1D_C55")
assert np.allclose(reread.spectra_values, steel_spectrum.spectra_values, rtol=1e-2), (
    "the fluxes file of cell 55 does not contain its spectrum"
)

text = steel_file.read_text()
for keyword in ("GETXS 0", "GETDECAY 0", "MASS 1.0", "DOSE 1", "UNCERT 2"):
    assert keyword in text, f"{keyword!r} is missing from {steel_file.name}"
iron = [float(line.split()[1]) for line in text.splitlines() if line.startswith("Fe ")]
assert iron and iron[0] > 50, "the MASS block should hold the steel: mostly iron"

written = IrradiationScenario.from_fispact(steel_file)
assert len(written.pulses) == len(pulsed.pulses), (
    f"{len(written.pulses)} pulses in the file, the pulsed scenario has {len(pulsed.pulses)}"
)
peak = max(p.intensity for p in written.pulses)
assert abs(peak / steel_flux - 1) < 1e-6, (
    f"the full-power pulses are written at {peak:.4e} n/cm2/s, expected {steel_flux:.4e}"
)
print("Exercise 1 correct")

Exercise 1 correct


The scenario was written once, in fractions of full power, and reused unchanged: only `norm`
moved from the flux of cell 53 to the flux of cell 55. That is the point of keeping the
intensity out of the scenario - one operating history, as many components as you need.

The other thing to notice is what was **not** reused: the collapse. The spectrum of cell 55 is
different, so it needs its own `fluxes` file and its own collapse run before this inventory
input can be run.

## Exercise 2 - dose or heat, same culprits?

The shutdown dose rate is a shielding problem, the decay heat is a cooling problem. Both come
out of the same inventory, but not necessarily out of the same nuclides. Check it on
`fispact_out`, **one day** after shutdown.

1. get the nuclides that make up 95% of the **dose** and those that make up 95% of the **decay
   heat**, both at the `"1d"` cooling time, and call them `dose_drivers` and `heat_drivers`;
2. label the rows of the two tables with a `nuclide` column built from the element, the isotope
   and the state, then compute the two set differences `dose_only` and `heat_only` and print
   them together with the nuclides common to both lists;
3. build a table `comparison` with one row per nuclide of the **union** of the two lists,
   holding the `nuclide` label, its `isotope % dose` and its `isotope % heat`, plus a
   `heat/dose` column with the ratio of the two shares;
4. print the nuclide with the largest ratio. Why is it a cooling problem and not a shielding
   problem?

**Hints**
* `filter_by_cum_dose(95, "1d")` and `filter_by_cum_heating(95, "1d")`; the label must be
  written exactly as it was given to `FispactOutput`;
* the state is `""` for a ground state and `"m"` for a metastable, so
  `df["element"] + df["isotope"].astype(str) + df["state"]` is a good label;
* the shares of **all** the nuclides, not only of the drivers, are in `fispact_out.sddr` and
  `fispact_out.decay_heat`: filter them on the cooling time, then keep your union of nuclides;
* `pd.merge(dose_part, heat_part, on="nuclide", how="outer")` puts the two share columns side
  by side.

In [4]:
label = "1d"


def add_nuclide(df: pd.DataFrame) -> pd.DataFrame:
    df["nuclide"] = df["element"] + df["isotope"].astype(str) + df["state"]
    return df


# 1: the two lists of drivers
dose_drivers = add_nuclide(fispact_out.filter_by_cum_dose(95, label).copy())
heat_drivers = add_nuclide(fispact_out.filter_by_cum_heating(95, label).copy())

# 2: who is in which list
dose_set = set(dose_drivers["nuclide"])
heat_set = set(heat_drivers["nuclide"])
dose_only = dose_set - heat_set
heat_only = heat_set - dose_set

print("both      :", sorted(dose_set & heat_set))
print("dose only :", sorted(dose_only))
print("heat only :", sorted(heat_only))

# 3: their two shares side by side, taken from the complete tables
union = dose_set | heat_set
dose_all = add_nuclide(fispact_out.sddr[fispact_out.sddr["cooling time"] == label].copy())
heat_all = add_nuclide(
    fispact_out.decay_heat[fispact_out.decay_heat["cooling time"] == label].copy()
)
comparison = pd.merge(
    dose_all[dose_all["nuclide"].isin(union)][["nuclide", "isotope % dose"]],
    heat_all[heat_all["nuclide"].isin(union)][["nuclide", "isotope % heat"]],
    on="nuclide",
    how="outer",
)
comparison["heat/dose"] = comparison["isotope % heat"] / comparison["isotope % dose"]
comparison = comparison.sort_values("heat/dose", ascending=False)

# 4: the one that heats more than it irradiates
worst = comparison.iloc[0]
print(
    f"\n{worst['nuclide']} is {worst['isotope % dose']:.2f}% of the dose "
    f"but {worst['isotope % heat']:.2f}% of the heat"
)
comparison

both      : ['Co58', 'Co60', 'Mn54', 'Ta182']
dose only : ['Ni57']
heat only : ['Co57', 'Cr51', 'Mo99']

Mo99 is 0.59% of the dose but 2.40% of the heat


,nuclide,isotope % dose,isotope % heat,heat/dose
5,Mo99,0.590541,2.398564,4.061640
0,Co57,0.733974,2.656097,3.618791
3,Cr51,0.664218,1.058566,1.593702
7,Ta182,3.711890,3.893906,1.049036
1,Co58,57.625344,55.998079,0.971761
6,Ni57,1.111886,1.029310,0.925733
4,Mn54,20.738961,18.913259,0.911968
2,Co60,12.628831,10.791683,0.854527


In [5]:
# --- self-check, do not modify ---
assert len(dose_drivers) == 5, f"the 95% dose table has {len(dose_drivers)} rows, expected 5"
assert len(heat_drivers) == 7, f"the 95% heat table has {len(heat_drivers)} rows, expected 7"
assert set(dose_only) == {"Ni57"}, f"the dose-only nuclide should be Ni57, you found {dose_only}"
assert set(heat_only) == {"Co57", "Cr51", "Mo99"}, (
    f"the heat-only nuclides should be Co57, Cr51 and Mo99, you found {heat_only}"
)
for column in ("nuclide", "isotope % dose", "isotope % heat", "heat/dose"):
    assert column in comparison.columns, (
        f"no {column!r} column in comparison, its columns are {list(comparison.columns)}"
    )
assert len(comparison) == 8, f"the comparison table has {len(comparison)} rows, expected 8"
worst = comparison.sort_values("heat/dose", ascending=False).iloc[0]
assert worst["nuclide"] == "Mo99", f"the largest heat/dose ratio is {worst['nuclide']}, not Mo99"
assert worst["heat/dose"] > 3.5, (
    f"Mo99 carries {worst['heat/dose']:.2f} times its dose share as heat, expected about 4"
)
print("Exercise 2 correct")

Exercise 2 correct


The big four - Co58, Mn54, Co60 and Ta182 - drive both quantities, but the edges of the two
lists differ. Ni57 only shows up in the dose; Co57, Cr51 and Mo99 only in the heat.

Mo99 is the extreme case: 0.6% of the dose but 2.4% of the heat. About three quarters of its
decay heat is **beta** energy, deposited in the component and never reaching the worker
standing next to it. Co57 is the same story told by soft gammas: its strongest line sits at
122 keV, absorbed close to where it is emitted.

Cooling and shielding are therefore not driven by the same nuclides, and a nuclide list built
for one of the two problems cannot be reused for the other.